# 2. Cleaning the data

Turns the raw file into tidy data that is safe to analyse. The cleaning code lives in `market_sales/cleaning.py`, so this notebook shows what it does and why, and the same steps run with `python -m market_sales.clean`.

**This is sample data**, so what it shows about shoppers isn't about real customers.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd

from market_sales.cleaning import DROPPED, clean_sales, find_problems
from market_sales.config import RAW_DATA

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 30)

## The raw file

The date and time are stored as text in two separate columns, and some column names are hard to use in code (`Tax 5%`, `Unit price`).

In [2]:
raw = pd.read_csv(RAW_DATA)
print(f"{len(raw):,} rows, {raw.shape[1]} columns")
raw.head(3)

1,000 rows, 17 columns


,Invoice ID,Branch,City,Customer type,Gender,Product line,Unit price,Quantity,Tax 5%,Total,Date,Time,Payment,cogs,gross margin percentage,gross income,Rating
0,750-67-8428,A,Yangon,Member,Female,Health and beauty,74.69,7,26.1415,548.9715,1/5/2019,13:08,Ewallet,522.83,4.761905,26.1415,9.1
1,226-31-3081,C,Naypyitaw,Normal,Female,Electronic accessories,15.28,5,3.8200,80.2200,3/8/2019,10:29,Cash,76.40,4.761905,3.8200,9.6
2,631-41-3108,A,Yangon,Normal,Male,Home and lifestyle,46.33,7,16.2155,340.5255,3/3/2019,13:23,Credit card,324.31,4.761905,16.2155,7.4


In [3]:
raw.dtypes

Invoice ID                     str
Branch                         str
City                           str
Customer type                  str
Gender                         str
Product line                   str
Unit price                 float64
Quantity                     int64
Tax 5%                     float64
Total                      float64
Date                           str
Time                           str
Payment                        str
cogs                       float64
gross margin percentage    float64
gross income               float64
Rating                     float64
dtype: object

## Problems found

1. **Date and time are text.** `Date` (like `1/5/2019`) and `Time` (like `13:08`) can't be sorted, grouped by weekday or filtered until they're real timestamps. The month is also ambiguous: `1/5/2019` is 5 January in this file, not 1 May.
2. **Misleading names.** `cogs` sounds like the shop's cost of the goods, and `gross income` sounds like profit. The next section shows they're neither.
3. **Columns that add nothing.** One is a copy of another, and one never changes.

### What the confusing columns really are

Each test below should print `True` if the column is what I claim it is.

In [4]:
tests = pd.Series({
    "cogs = unit price x quantity (so it is the price before tax)": (raw["cogs"] - raw["Unit price"] * raw["Quantity"]).abs().max() < 0.001,
    "Tax 5% = 5% of cogs": (raw["Tax 5%"] - raw["cogs"] * 0.05).abs().max() < 0.001,
    "Total = cogs + tax": (raw["Total"] - (raw["cogs"] + raw["Tax 5%"])).abs().max() < 0.001,
    "gross income = Tax 5% (so it is the tax, not profit)": (raw["gross income"] - raw["Tax 5%"]).abs().max() < 0.001,
    "gross margin percentage is the same in every row": raw["gross margin percentage"].nunique() == 1,
})
tests.to_frame("holds for every row")

,holds for every row
cogs = unit price x quantity (so it is the price before tax),True
Tax 5% = 5% of cogs,True
Total = cogs + tax,True
"gross income = Tax 5% (so it is the tax, not profit)",True
gross margin percentage is the same in every row,True


Tax is money collected for the government, so calling it `gross income` would overstate what the shop earns. The data has no real cost or profit figures, so profit can't be analysed at all.

`cogs` becomes **`subtotal`**, and `gross income` and `gross margin percentage` are dropped because they carry nothing the other columns don't. The cleaning code only drops them after proving the tests above, so if a future file broke those rules it would stop instead of dropping real information.

## Clean it

In [5]:
clean = clean_sales(raw)
clean.head(5)

,invoice_id,sold_at,date,month,month_name,weekday,is_weekend,hour,branch,city,customer_type,gender,product_line,unit_price,quantity,subtotal,tax,total,payment,rating
0,765-26-6951,2019-01-01 10:39:00,2019-01-01,1,January,Tuesday,False,10,A,Yangon,Normal,Male,Sports and travel,72.61,6,435.66,21.783,457.443,Credit card,6.9
1,746-04-1077,2019-01-01 11:36:00,2019-01-01,1,January,Tuesday,False,11,B,Mandalay,Member,Female,Food and beverages,84.63,10,846.30,42.315,888.615,Credit card,9.0
2,271-77-8740,2019-01-01 11:40:00,2019-01-01,1,January,Tuesday,False,11,C,Naypyitaw,Member,Female,Sports and travel,29.22,6,175.32,8.766,184.086,Ewallet,5.0
3,133-14-7229,2019-01-01 11:43:00,2019-01-01,1,January,Tuesday,False,11,C,Naypyitaw,Normal,Male,Health and beauty,62.87,2,125.74,6.287,132.027,Cash,5.0
4,651-88-7328,2019-01-01 13:55:00,2019-01-01,1,January,Tuesday,False,13,A,Yangon,Normal,Female,Fashion accessories,65.74,9,591.66,29.583,621.243,Cash,7.7


In [6]:
clean.dtypes

invoice_id                  str
sold_at          datetime64[us]
date             datetime64[us]
month                     int32
month_name             category
weekday                category
is_weekend                 bool
hour                      int32
branch                 category
city                   category
customer_type          category
gender                 category
product_line           category
unit_price              float64
quantity                  int64
subtotal                float64
tax                     float64
total                   float64
payment                category
rating                  float64
dtype: object

What changed:

- Names are lowercase with underscores.
- `sold_at` combines the date and time into one real timestamp, and the table is sorted by it.
- New columns for `month`, `weekday`, `hour` and `is_weekend`. Weekdays and months are ordered, so tables and charts show Monday to Sunday and January to March, not alphabetical order.
- Text categories use the `category` type, which saves memory and keeps their allowed values explicit.

In [7]:
print('Dropped:', DROPPED)
print('Renamed: cogs -> subtotal')
print('Rows before and after:', len(raw), len(clean))
print('Total sales before and after:', round(raw['Total'].sum(), 2), round(clean['total'].sum(), 2))

Dropped: ['gross income', 'gross margin percentage']
Renamed: cogs -> subtotal
Rows before and after: 1000 1000
Total sales before and after: 322966.75 322966.75


## Checking the result

Beyond the arithmetic above, the cleaning code checks that there are no missing values, every invoice ID is unique, each branch belongs to exactly one city, quantities are whole numbers of at least 1, prices are above zero and ratings are between 1 and 10.

In [8]:
problems = find_problems(clean)
print("Problems found:", problems if problems else "none")
summary = pd.Series({
    "rows": len(clean),
    "missing values": int(clean.isna().sum().sum()),
    "unique invoices": clean["invoice_id"].nunique(),
    "branches": clean["branch"].nunique(),
    "cities": clean["city"].nunique(),
})
summary.to_frame("count")

Problems found: none


,count
rows,1000
missing values,0
unique invoices,1000
branches,3
cities,3


## What time covers

A quick look at the new time columns, to make sure they behave.

In [9]:
span = clean["sold_at"].agg(["min", "max"])
print("First sale:", span["min"])
print("Last sale: ", span["max"])
print("Days with at least one sale:", clean["date"].nunique(), "of", (span["max"].normalize() - span["min"].normalize()).days + 1)
print("Opening hours seen: ", clean["hour"].min(), "to", clean["hour"].max())

First sale: 2019-01-01 10:39:00
Last sale:  2019-03-30 20:37:00
Days with at least one sale: 89 of 89
Opening hours seen:  10 to 20


In [10]:
clean.groupby('month_name', observed=True)['total'].agg(sales='sum', sales_count='count').round(0)

,sales,sales_count
month_name,,
January,116292.0,352
February,97219.0,303
March,109456.0,345


In [11]:
clean.groupby('weekday', observed=True)['total'].agg(sales='sum', sales_count='count').round(0)

,sales,sales_count
weekday,,
Monday,37899.0,125
Tuesday,51482.0,158
Wednesday,43731.0,143
Thursday,45349.0,138
Friday,43926.0,139
Saturday,56121.0,164
Sunday,44458.0,133


Weekdays come out in calendar order, which is the reason for the ordered categories.

## Next

This notebook only prepares the data. **Phase 3** asks better questions of it, such as which cities, products, days and hours earn the most money, and **Phase 4** tests whether any difference is bigger than chance.